In [1]:
import os
from google import genai
from google.genai import types
from tavily import TavilyClient
GEMINI_API_KEY = os.getenv('GEMINI_TOKEN')
GEMINI_MODEL = 'gemini-2.5-flash'
# Only run this block for Gemini Developer API
client = genai.Client(api_key=GEMINI_API_KEY)

In [2]:
prompt = 'Ciao!'

In [3]:
parts = [
        types.Part.from_text(text=prompt),
    ]
content_list = [
        types.Content(
            role='user',
            parts=parts
        )
]
res = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=content_list,
            config=types.GenerateContentConfig(
                temperature=0.3,
            )
    )
res.text

'Ciao! Come stai?'

In [3]:
class Agent:
    def __init__(self, system="", tools = dict()):
        self.system = system
        self.messages = []
        self.tools = tools

    def __call__(self, message):
        self.messages.append(
        types.Content(
            role='user',
            parts=[
        types.Part.from_text(text=message),
    ]
        ))
        result = self.execute()
        self.messages.append(types.Content(
            role='model',
            parts=[
        types.Part.from_text(text=result),
    ]
        ))
        return result

    def execute(self):
        
        while True:
            res = client.models.generate_content(
            model=GEMINI_MODEL,
            
            contents=self.messages,
            config=types.GenerateContentConfig(
                system_instruction=self.system,
                temperature=0.3,
                tools = list(self.tools.values()),
                automatic_function_calling=types.AutomaticFunctionCallingConfig(
                    disable=True
                )
            )
            )
            
            if res.function_calls:
                for call in res.function_calls:
                    try:
                        function_result = self.tools[call.name](
                            **call.args
                        )
                        function_response = {'result': function_result}
                        print('-------------------')
                        print(function_result)
                    except Exception as e:
                        function_response = {'error': str(e)}
                function_response_part = types.Part.from_function_response(
                name=call.name,
                response=function_response,
            )
            
                ##IMPORTANTE: DOBBIAMO FAR CAPIRE AL MOEDLLO CHE LA RISPOSTA è UNA TOOL CALL QUINDI IL ROLE DEVE ESSERE SETTATO A tool
                function_response_content = types.Content(
                    role='tool', parts=[function_response_part]
                )
                self.messages.append(function_response_content)
            else:
                break
        
        return res.text

In [4]:
def tavily_search_tool(
    query: str, max_results: int = 5
) -> list[dict]:
    """
    Perform a search using the Tavily API.

    Args:
        query (str): The search query.
        max_results (int): Number of results to return (default 5).
        include_images (bool): Whether to include image results.

    Returns:
        List[dict]: A list of dictionaries with keys like 'title', 'content', and 'url'.
    """
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        raise ValueError("TAVILY_API_KEY not found in environment variables.")

    client = TavilyClient(api_key)

    try:
        response = client.search(
            query=query, max_results=max_results
        )

        results = []
        for r in response.get("results", []):
            results.append(
                {
                    "title": r.get("title", ""),
                    "content": r.get("content", ""),
                    "url": r.get("url", ""),
                }
            )

        return results

    except Exception as e:
        return [{"error": str(e)}]  # For LLM-friendly agents

In [5]:
def sum_numbers(a: int, b:int) -> int:
    """Somma i due numeri a e b e restituisci il risultato"""
    return a + b

In [8]:
bot = Agent(system = 'Sei un assistente AI che ha a disposizione dei tool per cercare informazioni online per rispondere alle richieste dell utente. Oggi è il 22/04/2026', tools = {"tavily_search_tool": tavily_search_tool})

In [9]:
bot('Chi ha vinto il campionato di Calcio del 2025? Chi è stato il giocatore più pagato di quella squadra?')

-------------------
[{'title': 'Il Napoli è campione d’Italia per la quarta volta', 'content': 'Sin dalla seconda giornata, comunque, il Napoli ha cominciato a trovare la quadra ed è diventata una delle squadre più solide e competitive della Serie A, grazie al sistema di gioco molto preciso ed efficace di Antonio Conte e alla rigida disciplina che pretende dai suoi calciatori. Nel corso della stagione Conte è stato un po’ più flessibile del solito, alternando diversi sistemi di gioco (ha messo in campo la squadra con il 4-3-3, il 4-2-3-1, il 3-4-1-2, oltre che con il 3-5-2, il suo modulo di riferimento) e riuscendo a far fronte alla cessione di Kvaratskhelia a gennaio, forse non troppo rimpianta dallo stesso Conte. Ciononostante, anche grazie al pareggio ottenuto nei minuti finali nello scontro diretto contro l’Inter (finito 1-1, come all’andata), la squadra di Conte è riuscita a rimanere in corsa per il titolo, e con le quattro vittorie di fila ottenute da metà aprile in poi, seppur g

'Il campionato di calcio italiano del 2025 è stato vinto dal Napoli. Il giocatore più pagato di quella squadra è stato Romelu Lukaku, con 6 milioni di euro netti a stagione.'

In [17]:
bot('Adesso mi cerchi più informazioni sul primo evento?')

[FunctionCall(
  args={
    'query': 'Romics alla Fiera di Roma aprile 2026'
  },
  name='tavily_search_tool'
)]
-------------------
[{'title': 'Romics aprile 2026 - Fiere', 'content': 'Romics aprile 2026 · Dal giovedì 9 al domenica 12 aprile 2026 · Centro fieristico: Fiera di Roma · Città: Roma · Paese: Italia · Più info.: romics.it.', 'url': 'https://www.nfiere.com/romics/'}, {'title': 'Romics (Apr 2026), Rome Italy - Workshop - 10Times', 'content': 'Romics, a vibrant celebration of comics, animation, and gaming, returns to the Nuova Fiera di Roma from April 9 to April 12, 2026.', 'url': 'https://10times.com/romics'}, {'title': 'Romics, Festival Internazionale del Fumetto, Animazione, Cinema e ...', 'content': 'La 36esima edizione di Romics, il Festival Internazionale del Fumetto, Animazione, Cinema e Games si terrà alla Fiera di Roma dal 9 al 12 aprile', 'url': 'https://www.romatoday.it/eventi/romics-9-12-aprile-2026.html'}, {'title': 'Dal 9 al 12 aprile 2026 torna a Roma la magia d

"Romics, il Festival Internazionale del Fumetto, Animazione, Cinema e Games, si terrà alla Fiera di Roma dal **9 al 12 aprile 2026**.\n\nQuesta sarà la 36ª edizione dell'evento, che si conferma come uno degli appuntamenti più importanti in Italia per gli appassionati di questi settori. Romics offre mostre, incontri, proiezioni e spazi dedicati alla cultura pop, con un'attenzione particolare all'evoluzione del fumetto e dell'animazione, valorizzando il dialogo tra tradizione artistica e innovazione tecnologica. È un evento inclusivo, pensato per tutte le generazioni, che promuove il rinnovamento del linguaggio del fumetto e della narrazione visiva."

In [10]:
bot.messages

[Content(
   parts=[
     Part(
       text='Chi ha vinto il campionato di Calcio del 2025? Chi è stato il giocatore più pagato di quella squadra?'
     ),
   ],
   role='user'
 ),
 Content(
   parts=[
     Part(
       function_response=FunctionResponse(
         name='tavily_search_tool',
         response={
           'result': [
             {<... 3 items at Max depth ...>},
             {<... 3 items at Max depth ...>},
             {<... 3 items at Max depth ...>},
             {<... 3 items at Max depth ...>},
             {<... 3 items at Max depth ...>},
           ]
         }
       )
     ),
   ],
   role='tool'
 ),
 Content(
   parts=[
     Part(
       function_response=FunctionResponse(
         name='tavily_search_tool',
         response={
           'result': [
             {<... 3 items at Max depth ...>},
             {<... 3 items at Max depth ...>},
             {<... 3 items at Max depth ...>},
             {<... 3 items at Max depth ...>},
             {<... 3 i